# Jour 2 · Chapitre 5 — Scénario : le rapport mensuel du comité de direction
## Formation Beobank · SAS → Python · Cahier participant

**L'histoire.** Chaque mois, le comité de direction reçoit une vue « portefeuille par client ».
Jusqu'ici, Nathalie la produisait à la main dans Excel en recopiant 4 fichiers. Elle part en congé :
*"Peux-tu automatiser mon rapport ? Il faut relier les clients à leurs contrats, compter les
comptes par client, les soldes, un tableau croisé par statut, un graphique, et déposer le
résultat dans la base pour le tableau de bord."*

**Comment travailler.** Remplacez chaque `______`. Ce scénario combine **plusieurs tables** :
c'est la journée Pandas en condensé.

📎 **Matière à relire :** `Jour2_Formateur.ipynb` (même dossier) — Modules 3 (`.loc`/`.iloc`),
4 (filtres), 5 (colonnes calculées), 6 (`groupby`), 7 (`merge`), 8 (`pivot_table`, `apply`),
9 (Matplotlib) et 11 (export et SQLite).

## Préparation (déjà écrite, exécutez-la)

In [ ]:
# ── Imports et chargement ─────────────────────────────────────────────────
import os
import sqlite3
import matplotlib.pyplot as plt
import pandas as pd

DATA = "../../data/"
PARAMS = dict(sep=";", na_values=".", encoding="utf-8")
ctr  = pd.read_csv(DATA + "CTR.csv", **PARAMS)
tie  = pd.read_csv(DATA + "TIE.csv", **PARAMS)
lien = pd.read_csv(DATA + "TIE_X_CTR.csv", **PARAMS)
print(ctr.shape, tie.shape, lien.shape)

---
## Mission 1 — Repérer les lignes utiles

*"Montre-moi la première ligne du fichier contrats, puis les comptes qui ne sont pas en euros."*

In [ ]:
# ── .iloc (par position) et .loc (par condition + colonnes) ───────────────
# Quel accesseur sélectionne par POSITION (numéro de ligne) ?
print(ctr.______[0])
# Quel accesseur sélectionne par condition ET par noms de colonnes ?
ctr.______[ctr["COD_DEV"] != "EUR", ["IDT_AC", "COD_DEV", "SLD_CTR"]]

In [ ]:
# ── Filtre sur une liste de valeurs ───────────────────────────────────────
# Quelle méthode teste l'appartenance à une liste (plus lisible que trois « | ») ?
devises_etrangeres = ctr[ctr["COD_DEV"].______(["USD", "AUD", "NOK"])]
print("Comptes en devises :", len(devises_etrangeres))          # → 7

---
## Mission 2 — Enrichir les contrats

*"Le comité ne lit pas les codes : mets des libellés, et l'année d'ouverture."*

In [ ]:
# ── Colonnes calculées ────────────────────────────────────────────────────
# Quelle méthode applique un dictionnaire de correspondance code → libellé ?
ctr["STATUT"] = ctr["COD_ECV_CTR"].______({4: "Clôturé", 6: "Résilié"})
# Quel accesseur des dates donne l'année ?
ctr["ANNEE_OUV"] = pd.to_datetime(ctr["DAT_OUV_CTR"]).______.year
ctr[["IDT_AC", "STATUT", "ANNEE_OUV"]].head(3)

---
## Mission 3 — Relier clients et contrats

`TIE_X_CTR` relie un client (`IDT_PI`) à un compte (`IDT_AC`).
On construit : **lien → contrats → clients**.

In [ ]:
# ── Jointures ─────────────────────────────────────────────────────────────
colonnes_ctr = ["IDT_AC", "COD_DEV", "SLD_CTR", "STATUT", "ANNEE_OUV"]
# Quelle méthode joint deux DataFrames ? Sur quelle clé relier un lien à un contrat ?
vue = lien.______(ctr[colonnes_ctr], on="______", how="left")
# Quelle clé relie ensuite au client ? On garde tous les liens : quel type de jointure ?
vue = vue.merge(tie[["IDT_PI", "COD_LNG_CTR"]], on="______", how="______")
print(vue.shape)                                                  # → (200, 11)
vue.head(3)

---
## Mission 4 — Le portefeuille par client

*"Pour chaque client : nombre de comptes, nombre de soldes connus, somme des soldes."*

In [ ]:
# ── groupby + agg nommée ──────────────────────────────────────────────────
# Quelle méthode regroupe par client ? Quelle méthode applique plusieurs calculs nommés ?
portefeuille = vue.______("NUM_TIE").______(
    nb_comptes=("IDT_AC", "size"),
    # Quelle fonction compte les valeurs NON manquantes ?
    nb_soldes=("SLD_CTR", "______"),
    # Quelle fonction additionne ?
    solde_total=("SLD_CTR", "______"),
    langue=("COD_LNG_CTR", "first"),
)
portefeuille.sort_values("nb_comptes", ascending=False)

---
## Mission 5 — Le tableau croisé par statut

In [ ]:
# ── pivot_table ───────────────────────────────────────────────────────────
# Quelle fonction pandas crée un tableau croisé ? Quelle valeur mettre dans les cases vides ?
croise = pd.______(vue, index="NUM_TIE", columns="STATUT", values="IDT_AC",
                           aggfunc="count", fill_value=______)
croise

---
## Mission 6 — Segmenter les clients

*"Au-delà de 20 comptes, c'est un client « multi-équipé », sinon « standard »."*

In [ ]:
# ── apply + lambda ────────────────────────────────────────────────────────
# Quelle méthode applique une fonction à chaque valeur d'une colonne ?
portefeuille["segment"] = portefeuille["nb_comptes"].______(
    lambda n: "Multi-équipé" if n >= 20 else "Standard")
portefeuille["segment"].value_counts()                            # → Multi-équipé 5, Standard 5

---
## Mission 7 — Le graphique du comité

In [ ]:
# ── Barres horizontales ───────────────────────────────────────────────────
# Quel kind= dessine des barres HORIZONTALES (pratique pour de longs identifiants) ?
ax = portefeuille["nb_comptes"].sort_values().plot(kind="______", color="#2E5A88")
ax.set_title("Nombre de comptes par client")
ax.set_xlabel("Comptes")
plt.show()

---
## 🏁 Mission finale — Livrer le rapport

*"Un CSV pour le comité, et une table dans la base pour le tableau de bord."*

In [ ]:
# ── Export CSV ────────────────────────────────────────────────────────────
os.makedirs("../../output", exist_ok=True)
# Quelle méthode écrit un fichier CSV ?
portefeuille.______("../../output/rapport_portefeuille.csv", sep=";")

# ── Dépôt dans la base SQL ────────────────────────────────────────────────
conn = sqlite3.connect(":memory:")
# NUM_TIE est l'index : quelle méthode le remet en colonne ? Quelle méthode écrit une table SQL ?
portefeuille.______().______("PORTEFEUILLE", conn, index=False)
# Le tableau de bord lit le top 3 : quelle fonction pandas exécute une requête SQL ?
pd.______("SELECT NUM_TIE, nb_comptes, solde_total FROM PORTEFEUILLE "
               "ORDER BY nb_comptes DESC LIMIT 3", conn)

✍️ **Message au comité (2 phrases) :** que retenez-vous de la concentration des comptes sur
quelques clients ?